In [1]:
# ===== P7 CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== P7 CELL 2: Load everything needed for the statistics =====
import os, json, hashlib, math, time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
FEAT_DIR = os.path.join(BASE, "02_data_processed", "features")
SPLIT_DIR = os.path.join(BASE, "02_data_processed", "splits")
SCORE_DIR = os.path.join(BASE, "04_scores")
RES_DIR = os.path.join(BASE, "05_results")

with open(os.path.join(BASE, "config.json")) as fh:
    CONFIG = json.load(fh)
proto = CONFIG["protocol"]
with open(os.path.join(BASE, proto["file"]), "rb") as fh:
    assert hashlib.sha256(fh.read()).hexdigest() == proto["sha256"], "Protocol file changed - STOP"
PLAN = proto["plan"]
A0, B = PLAN["primary_alpha"], PLAN["bootstrap"]["resamples"]
print("Protocol", proto["version"], "verified unchanged")

meta_val = pd.read_pickle(os.path.join(FEAT_DIR, "meta_val.pkl"))
meta_test = pd.read_pickle(os.path.join(FEAT_DIR, "meta_test.pkl"))
y = meta_test["evil"].values.astype(int)
CONFIGS = sorted(f[:-4] for f in os.listdir(SCORE_DIR) if f.endswith(".npz"))
SCORES = {c: dict(np.load(os.path.join(SCORE_DIR, c + ".npz"))) for c in CONFIGS}

PROPOSED = ["iforest__F_main__s42", "autoencoder__F_main__s42"]
BASELINES = ["random__none__s42", "rule_userId__none__s0", "rarity__none__s0", "ocsvm__F_main__s42"]
PRIMARY = PROPOSED + BASELINES

def conformal_p(cal_scores, new_scores):
    cal = np.sort(np.asarray(cal_scores))
    n_ge = len(cal) - np.searchsorted(cal, np.asarray(new_scores), side="left")
    return (1 + n_ge) / (len(cal) + 1)

# process index of every test row (the bootstrap resamples whole processes)
proc_codes, proc_keys = pd.factorize(meta_test["process_key"], sort=True)
n_proc = len(proc_keys)
proc_evil = np.bincount(proc_codes, weights=y, minlength=n_proc) > 0
print(f"Test: {n_proc} processes ({proc_evil.sum()} with attack events). Bootstrap resamples: {B}")

Protocol v1 verified unchanged
Test: 198 processes (121 with attack events). Bootstrap resamples: 1000


In [3]:
# ===== P7 CELL 3: Per-configuration quantities that the bootstrap reweights =====
def prepare(cfg):
    s_val, s_test = SCORES[cfg]["val"], SCORES[cfg]["test"]
    kept = conformal_p(s_val, s_test) <= A0
    # event level, summed per process
    ev_kept_att = np.bincount(proc_codes, weights=kept & (y == 1), minlength=n_proc)
    ev_att = np.bincount(proc_codes, weights=(y == 1), minlength=n_proc)
    ev_kept_norm = np.bincount(proc_codes, weights=kept & (y == 0), minlength=n_proc)
    ev_norm = np.bincount(proc_codes, weights=(y == 0), minlength=n_proc)
    # process level
    pv = pd.Series(s_val).groupby(meta_val["process_key"].values).max().values
    ps = pd.Series(s_test).groupby(proc_codes).max().reindex(range(n_proc)).values
    p_kept = conformal_p(pv, ps) <= A0
    return {"s": s_test, "ev": (ev_kept_att, ev_att, ev_kept_norm, ev_norm), "ps": ps, "p_kept": p_kept}

def metrics(prep, w):
    """All bootstrap metrics for one resample; w = how many times each process was drawn."""
    ka, a, kn, n = prep["ev"]
    row_w = w[proc_codes]
    out = {"recall_event": (w * ka).sum() / (w * a).sum(),
           "FRR_event": (w * kn).sum() / max((w * n).sum(), 1e-12),
           "AUROC_event": roc_auc_score(y, prep["s"], sample_weight=row_w),
           "recall_process": (w * (prep["p_kept"] & proc_evil)).sum() / (w * proc_evil).sum(),
           "AUROC_process": roc_auc_score(proc_evil, prep["ps"], sample_weight=w)}
    return out

PREP = {c: prepare(c) for c in PRIMARY}
point = pd.DataFrame({c: metrics(PREP[c], np.ones(n_proc)) for c in PRIMARY}).T
print("Point estimates (should match Phase 6):")
print(point.round(4).to_string())

Point estimates (should match Phase 6):
                          recall_event  FRR_event  AUROC_event  recall_process  AUROC_process
iforest__F_main__s42            0.0002     0.0241       0.8213             0.0         0.5880
autoencoder__F_main__s42        1.0000     0.4133       0.9891             0.0         0.6704
random__none__s42               0.0098     0.0105       0.4993             0.0         0.4427
rule_userId__none__s0           0.9928     0.1296       0.9316             0.0         0.8465
rarity__none__s0                0.9949     0.4150       0.8024             0.0         0.6441
ocsvm__F_main__s42              1.0000     0.4588       0.9921             0.0         0.7193


In [4]:
# ===== P7 CELL 4: Cluster bootstrap over processes (1,000 resamples; about 5-10 minutes) =====
rng = np.random.default_rng(PLAN["random_seed"])
METRICS = ["recall_event", "FRR_event", "AUROC_event", "recall_process", "AUROC_process"]
boot = {c: {m: np.empty(B) for m in METRICS} for c in PRIMARY}
t0 = time.time()
for b in range(B):
    w = np.bincount(rng.integers(0, n_proc, n_proc), minlength=n_proc).astype(float)
    if (w * proc_evil).sum() == 0 or (w * ~proc_evil).sum() == 0:
        w = np.ones(n_proc)                       # practically never happens; keeps both classes present
    for c in PRIMARY:
        for m, v in metrics(PREP[c], w).items():
            boot[c][m][b] = v
    if (b + 1) % 100 == 0:
        print(f"  {b + 1}/{B} resamples done ({time.time() - t0:.0f}s)")
np.savez_compressed(os.path.join(RES_DIR, "P7_bootstrap_samples.npz"),
                    **{f"{c}|{m}": boot[c][m] for c in PRIMARY for m in METRICS})

ci_rows = []
for c in PRIMARY:
    for m in METRICS:
        lo, hi = np.percentile(boot[c][m], [2.5, 97.5])
        ci_rows.append({"config": c, "metric": m, "estimate": point.loc[c, m], "ci_low": lo, "ci_high": hi})
ci = pd.DataFrame(ci_rows)
ci.to_csv(os.path.join(RES_DIR, "P7_confidence_intervals.csv"), index=False)
table = ci.assign(text=lambda d: d.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1))
print("\nEstimate [95% CI]:")
print(table.pivot(index="config", columns="metric", values="text").loc[PRIMARY, METRICS].to_string())

  100/1000 resamples done (22s)
  200/1000 resamples done (39s)
  300/1000 resamples done (57s)
  400/1000 resamples done (75s)
  500/1000 resamples done (92s)
  600/1000 resamples done (109s)
  700/1000 resamples done (126s)
  800/1000 resamples done (143s)
  900/1000 resamples done (161s)
  1000/1000 resamples done (179s)

Estimate [95% CI]:
metric                            recall_event             FRR_event           AUROC_event        recall_process         AUROC_process
config                                                                                                                                
iforest__F_main__s42      0.000 [0.000, 0.004]  0.024 [0.006, 0.056]  0.821 [0.443, 0.941]  0.000 [0.000, 0.000]  0.588 [0.497, 0.681]
autoencoder__F_main__s42  1.000 [1.000, 1.000]  0.413 [0.186, 0.657]  0.989 [0.903, 0.995]  0.000 [0.000, 0.000]  0.670 [0.577, 0.759]
random__none__s42         0.010 [0.008, 0.010]  0.011 [0.009, 0.012]  0.499 [0.494, 0.504]  0.000 [0.000, 0.000]  

In [5]:
# ===== P7 CELL 5: Hypotheses H2 and H3 - paired differences, bootstrap p-values, Holm correction =====
tests = []
for prop in PROPOSED:
    for base in BASELINES:
        for metric, hyp in [("recall_event", "H2a"), ("recall_process", "H2b"),
                            ("AUROC_event", "H3-event"), ("AUROC_process", "H3-process")]:
            d = boot[prop][metric] - boot[base][metric]
            diff = point.loc[prop, metric] - point.loc[base, metric]
            p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
            if np.all(d == 0):
                p = 1.0
            lo, hi = np.percentile(d, [2.5, 97.5])
            tests.append({"hypothesis": hyp, "proposed": prop, "baseline": base, "metric": metric,
                          "difference": diff, "ci_low": lo, "ci_high": hi, "p_bootstrap": p})
tests = pd.DataFrame(tests)

# Holm correction across all 32 comparisons
order = np.argsort(tests["p_bootstrap"].values)
m = len(tests)
adj = np.empty(m)
running = 0.0
for rank, idx in enumerate(order):
    running = max(running, (m - rank) * tests["p_bootstrap"].values[idx])
    adj[idx] = min(1.0, running)
tests["p_holm"] = adj

def verdict(r):
    if r.p_holm < 0.05 and r.ci_low > 0:
        return "proposed better (supported)"
    if r.p_holm < 0.05 and r.ci_high < 0:
        return "baseline better"
    return "no significant difference"
tests["verdict"] = tests.apply(verdict, axis=1)
tests.to_csv(os.path.join(RES_DIR, "P7_hypothesis_tests_holm.csv"), index=False)
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 100)
print(f"{m} pre-registered comparisons (Holm-corrected):\n")
print(tests[["hypothesis", "proposed", "baseline", "difference", "ci_low", "ci_high", "p_holm", "verdict"]]
      .round(4).to_string(index=False))
print(f"\nNote: p = 0.000 means no resample went the other way (p below the resolution of {B} resamples).")
print("\nCounts of verdicts per hypothesis:")
print(pd.crosstab(tests["hypothesis"], tests["verdict"]).to_string())

32 pre-registered comparisons (Holm-corrected):

hypothesis                 proposed              baseline  difference  ci_low  ci_high  p_holm                     verdict
       H2a     iforest__F_main__s42     random__none__s42     -0.0097 -0.0098  -0.0048   0.000             baseline better
       H2b     iforest__F_main__s42     random__none__s42      0.0000  0.0000   0.0000   1.000   no significant difference
  H3-event     iforest__F_main__s42     random__none__s42      0.3219 -0.0591   0.4427   1.000   no significant difference
H3-process     iforest__F_main__s42     random__none__s42      0.1452  0.0419   0.2447   0.152   no significant difference
       H2a     iforest__F_main__s42 rule_userId__none__s0     -0.9927 -0.9980  -0.8273   0.000             baseline better
       H2b     iforest__F_main__s42 rule_userId__none__s0      0.0000  0.0000   0.0000   1.000   no significant difference
  H3-event     iforest__F_main__s42 rule_userId__none__s0     -0.1104 -0.3909  -0.0131   0

In [6]:
# ===== P7 CELL 6: Seed spread (5 seeds on F_main) =====
seed_rows = []
for model in ["iforest", "ocsvm", "autoencoder"]:
    for seed in PLAN["model_seeds"]["repeats"]:
        c = f"{model}__F_main__s{seed}"
        s_val, s_test = SCORES[c]["val"], SCORES[c]["test"]
        kept = conformal_p(s_val, s_test) <= A0
        seed_rows.append({"model": model, "seed": seed, "AUROC_event": roc_auc_score(y, s_test),
                          "recall_event": kept[y == 1].mean(), "FRR_event": kept[y == 0].mean()})
seeds = pd.DataFrame(seed_rows)
seeds.to_csv(os.path.join(RES_DIR, "P7_seed_results.csv"), index=False)
spread = seeds.groupby("model")[["AUROC_event", "recall_event", "FRR_event"]].agg(["mean", "std", "min", "max"])
spread.to_csv(os.path.join(RES_DIR, "P7_seed_spread.csv"))
print("Spread across 5 seeds (F_main, alpha = 0.01):")
print(spread.round(4).to_string())

Spread across 5 seeds (F_main, alpha = 0.01):
            AUROC_event                         recall_event                      FRR_event                        
                   mean     std     min     max         mean     std  min     max      mean     std     min     max
model                                                                                                              
autoencoder      0.9814  0.0099  0.9701  0.9891          1.0  0.0000  1.0  1.0000    0.4136  0.0014  0.4114  0.4149
iforest          0.8022  0.0351  0.7583  0.8505          0.0  0.0001  0.0  0.0002    0.0277  0.0127  0.0192  0.0501
ocsvm            0.9904  0.0011  0.9889  0.9921          1.0  0.0000  1.0  1.0000    0.4481  0.0291  0.4201  0.4936


In [7]:
# ===== P7 CELL 7: Secondary robustness check - real-record mixes (protocol section 8) =====
mixes = np.load(os.path.join(SPLIT_DIR, "secondary_mixes.npz"))
K = PLAN["secondary_mix"]["k_values"]
mix_rows = []
for key in mixes.files:
    kind, pi, seed = key.split("|")
    idx = mixes[key]
    ym = y[idx]
    for c in PRIMARY:
        s = SCORES[c]["test"][idx]
        tie = np.random.default_rng(42).random(len(s))
        order = np.lexsort((tie, -s))
        kept = conformal_p(SCORES[c]["val"], s) <= A0
        row = {"sampling": kind, "attack_share": float(pi.split("=")[1]), "seed": int(seed.split("=")[1]),
               "config": c, "PR_AUC": average_precision_score(ym, s), "no_skill_PR_AUC": ym.mean(),
               "precision_in_queue": ym[kept].mean() if kept.any() else np.nan}
        for k in K:
            row[f"precision_at_{k}"] = ym[order[:k]].mean()
        mix_rows.append(row)
mix = pd.DataFrame(mix_rows)
mix.to_csv(os.path.join(RES_DIR, "P7_secondary_mixes_all.csv"), index=False)
cols = ["PR_AUC", "no_skill_PR_AUC"] + [f"precision_at_{k}" for k in K] + ["precision_in_queue"]
agg = mix.groupby(["sampling", "attack_share", "config"])[cols].mean().round(3)
agg.to_csv(os.path.join(RES_DIR, "P7_secondary_mixes_summary.csv"))
print("Mean over 10 seeds (random-row mixes):")
print(agg.loc["random_rows"].to_string())
print("\nMean over 10 seeds (contiguous time-window mixes):")
print(agg.loc["contiguous_time_window"].to_string())

Mean over 10 seeds (random-row mixes):
                                       PR_AUC  no_skill_PR_AUC  precision_at_100  precision_at_1000  precision_in_queue
attack_share config                                                                                                    
0.01         autoencoder__F_main__s42   0.466             0.01             0.000              0.303               0.024
             iforest__F_main__s42       0.053             0.01             0.000              0.000               0.000
             ocsvm__F_main__s42         0.606             0.01             0.003              0.306               0.022
             random__none__s42          0.011             0.01             0.015              0.011               0.012
             rarity__none__s0           0.025             0.01             0.010              0.021               0.024
             rule_userId__none__s0      0.072             0.01             0.080              0.072               0.072
0